# SST-2 — result figures

Regenerates every SST-2 figure **from the saved result files**. Nothing is retrained here; this
notebook only reads, checks and plots.

Set `RESULTS_DIR` in the setup cell to wherever your result files live. The loader searches that
directory recursively and reads `.json`, `.xlsx` and `.csv` interchangeably, so it does not matter
which format each stage was saved in.

**Files this notebook looks for** — matched loosely by filename keywords, so
`e2_results_sst2.json`, `E2_sst2.xlsx` and `results/e2_sst2.csv` all work:

| Stage | Filename must contain | Provides |
|---|---|---|
| E1 | `e1` + `sst2` | clean baseline |
| E2 | `e2` + `sst2` | random-poisoned teachers |
| E3 | `e3` + `sst2` | CBS-poisoned teachers |
| E4 | `e4` + `sst2` | clean distilled student (control) |
| E5 | `e5` + `sst2` | students distilled from random teachers |
| E6 | `e6` + `sst2` | students distilled from CBS teachers |
| E7 | `e7` + `sst2` | defense table |
| Sweep | `validation` or `sweep` + `sst2` | poison-rate sweep |


In [4]:
!pip install pandas matplotlib openpyxl --quiet


In [5]:
"""
Flexible result loader.

Point RESULTS_DIR at wherever your result files live. The loader searches recursively,
matches files by filename keywords, and reads .json / .xlsx / .csv transparently -- so it
does not matter which format each stage happened to be saved in.
"""
import os, json, glob
import pandas as pd
import numpy as np

RESULTS_DIR = "./results"      # <-- EDIT THIS if your results live elsewhere
DATASET     = "sst2"          # sst2 | agnews | imdb | yelp

def _read_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".json":
        with open(path) as f:
            return json.load(f)
    if ext in (".xlsx", ".xls"):
        book = pd.read_excel(path, sheet_name=None)
        if len(book) == 1:
            return list(book.values())[0].to_dict(orient="records")
        return {name: df.to_dict(orient="records") for name, df in book.items()}
    if ext == ".csv":
        return pd.read_csv(path).to_dict(orient="records")
    raise ValueError(f"unsupported result file: {path}")

def find(*keywords, required=True):
    hits = []
    for path in glob.glob(os.path.join(RESULTS_DIR, "**", "*"), recursive=True):
        if not os.path.isfile(path):
            continue
        name = os.path.basename(path).lower()
        if os.path.splitext(name)[1] not in (".json", ".xlsx", ".xls", ".csv"):
            continue
        if all(k.lower() in name for k in keywords):
            hits.append(path)
    if not hits:
        if required:
            raise FileNotFoundError(
                f"No result file matching {keywords} under {RESULTS_DIR}. "
                f"Check RESULTS_DIR, or rename the file so its name contains those keywords.")
        return None
    hits.sort(key=len)
    return hits[0]

def load(*keywords, required=True):
    path = find(*keywords, required=required)
    if path is None:
        return None
    print(f"  loaded {os.path.relpath(path, RESULTS_DIR)}")
    return _read_any(path)

# --- normalisers: stages were saved in slightly different shapes ---
def as_trigger_dict(obj):
    """Coerce an E2/E3/E5/E6-style result into {'word': {...}, 'sent': {...}}."""
    if isinstance(obj, list):
        keyed = {}
        for row in obj:
            label = str(row.get("Unnamed: 0") or row.get("index") or row.get("config") or "").lower()
            keyed[label] = row
        obj = keyed
    if isinstance(obj, dict) and "word" in obj and "sent" in obj:
        return obj
    if isinstance(obj, dict) and "CACC" in obj and isinstance(obj["CACC"], dict):
        cols = list(obj.keys())
        rows = list(obj[cols[0]].keys())
        out = {}
        for r in rows:
            key = "word" if "word" in str(r).lower() else "sent"
            out[key] = {c: obj[c][r] for c in cols}
        return out
    out = {}
    for k, v in obj.items():
        key = "word" if "word" in str(k).lower() else "sent"
        out[key] = v
    return out

def as_flat(obj):
    """Coerce an E1/E4-style single-row result into a flat dict."""
    if isinstance(obj, list):
        return obj[0]
    if isinstance(obj, dict) and all(isinstance(v, dict) for v in obj.values()):
        inner = list(obj.values())[0]
        if set(str(k) for k in inner.keys()) <= {"0"}:
            return {k: list(v.values())[0] for k, v in obj.items()}
        return list(obj.values())[0]
    return obj

def as_defense_table(obj):
    """Coerce an E7 result into {config: {defense: value}} whatever the source format."""
    CONFIGS = ["WordInsert+Random","WordInsert+CBS","InsertSent+Random","InsertSent+CBS"]
    if isinstance(obj, dict) and all(c in obj for c in CONFIGS):
        return obj                                   # already nested (json)
    if isinstance(obj, dict):                        # multi-sheet workbook
        obj = list(obj.values())[0]
    if isinstance(obj, list):                        # rows: defense name + one column per config
        out = {c: {} for c in CONFIGS}
        for row in obj:
            name = row.get("Unnamed: 0") or row.get("index") or row.get("Defense")
            for c in CONFIGS:
                if c in row:
                    out[c][str(name)] = float(row[c])
        return out
    raise ValueError("unrecognised defense-table shape")

def sweep_frame(obj):
    """Coerce a sweep result into a DataFrame with poison_rate / ASR / trigger / method."""
    if isinstance(obj, dict):
        for key in ("sweep", "sweep_gaps", "Sheet1"):
            if key in obj:
                obj = obj[key]
                break
        else:
            obj = list(obj.values())[0]
    df = pd.DataFrame(obj)
    df = df[[c for c in df.columns if not str(c).startswith("Unnamed")]]
    return df.sort_values("poison_rate")

PALETTE = {"random": "#1f77b4", "cbs": "#d62728"}


AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

## Load every stage

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 130, "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 9, "axes.titlesize": 10})

print("Loading SST-2 results from", os.path.abspath(RESULTS_DIR))
E1 = as_flat(load("e1", DATASET))
E2 = as_trigger_dict(load("e2", DATASET))
E3 = as_trigger_dict(load("e3", DATASET))
E4 = as_flat(load("e4", DATASET))
E5 = as_trigger_dict(load("e5", DATASET))
E6 = as_trigger_dict(load("e6", DATASET))
E7 = as_defense_table(load("e7", DATASET))

_sw = load("validation", DATASET, required=False) or load("sweep", DATASET, required=False)
SWEEP = sweep_frame(_sw) if _sw is not None else None
print("\nsweep rows:", 0 if SWEEP is None else len(SWEEP))

## Sanity checks

Run these before trusting any plot. Each restates something the paper claims, so if a number
changed upstream it surfaces here rather than silently inside a figure.

In [ ]:
def chk(label, ok, detail=""):
    print(("  OK   " if ok else "  FAIL ") + label + (("  " + detail) if detail else ""))
    return ok

print("Clean baseline CACC: {:.2f}%".format(E1["CACC"]*100))
print()
print("Teachers")
for trig in ("word","sent"):
    for tag, blk in (("Random", E2), ("CBS", E3)):
        r = blk[trig]
        chk("{:6s} {:5s} ASR={:6.2f}%  negctrl={:5.2f}%".format(
                tag, trig, r["ASR"]*100, r["ASR_negctrl"]*100),
            r["ASR"] > r["ASR_negctrl"]*2,
            "(ASR must sit far above its negative control)")

print()
print("Distilled students -- retention = student ASR / teacher ASR")
for trig in ("word","sent"):
    rr = E5[trig]["ASR"]/E2[trig]["ASR"]
    rc = E6[trig]["ASR"]/E3[trig]["ASR"]
    print("  {:5s}  Random {:6.2f}%   CBS {:6.2f}%   CBS retains {}".format(
        trig, rr*100, rc*100, "more" if rc > rr else "less"))

print()
print("Clean-student control (E4): ASR = {:.2f}%  (should sit near the model's error rate)"
      .format(E4["ASR"]*100))

## Figure — poison-rate sweep

In [ ]:
if SWEEP is None:
    print("no sweep file found; skipping")
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
    for ax, trig in zip(axes, ("word","sent")):
        for meth in ("random","cbs"):
            sub = SWEEP[(SWEEP.trigger==trig) & (SWEEP.method==meth)]
            if len(sub)==0: continue
            ax.plot(sub.poison_rate*100, sub.ASR*100,
                    "o-" if meth=="random" else "s--", color=PALETTE[meth], ms=4, lw=1.4,
                    label="Random" if meth=="random" else "CBS")
        ax.axhline(90, color="gray", ls=":", lw=1)
        ax.set_xscale("log"); ax.set_ylim(-4,106)
        ax.set_xlabel("Poison rate (%, log)")
        ax.set_title("SST-2 — " + ("Word-insert" if trig=="word" else "InsertSent") + " trigger")
    axes[0].set_ylabel("Attack Success Rate (%)"); axes[0].legend(loc="lower right")
    plt.tight_layout(); plt.savefig("fig_sst2_sweep.png", dpi=200, bbox_inches="tight"); plt.show()

    print("\nPoison budget to reach 90% ASR")
    for trig in ("word","sent"):
        got = {}
        for meth in ("random","cbs"):
            sub = SWEEP[(SWEEP.trigger==trig)&(SWEEP.method==meth)&(SWEEP.ASR>=0.9)]
            got[meth] = sub.poison_rate.min() if len(sub) else None
        r, c = got["random"], got["cbs"]
        ratio = "{:.1f}x".format(c/r) if (r and c) else "CBS never reached 90%"
        print("  {:5s}  Random={}  CBS={}  ->  {}".format(trig, r, c, ratio))

## Figure — distillation retention

In [ ]:
labels, rv, cv = [], [], []
for trig in ("word","sent"):
    labels.append(trig)
    rv.append(E5[trig]["ASR"]/E2[trig]["ASR"]*100)
    cv.append(E6[trig]["ASR"]/E3[trig]["ASR"]*100)

fig, ax = plt.subplots(figsize=(5.2, 3.4))
x = np.arange(len(labels)); w = 0.36
b1 = ax.bar(x-w/2, rv, w, label="Random", color=PALETTE["random"])
b2 = ax.bar(x+w/2, cv, w, label="CBS", color=PALETTE["cbs"])
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("ASR retention (student / teacher, %)")
ax.set_title("SST-2: backdoor surviving clean-data distillation")
ax.legend()
for bars in (b1,b2):
    for bar in bars:
        ax.annotate("{:.2f}".format(bar.get_height()),
                    (bar.get_x()+bar.get_width()/2, bar.get_height()),
                    textcoords="offset points", xytext=(0,3), ha="center", fontsize=8)
plt.tight_layout(); plt.savefig("fig_sst2_retention.png", dpi=200, bbox_inches="tight"); plt.show()

## Figure — defenses

In [ ]:
DEFENSES = ["No defense","ONION","Spectral Signature","STRIP","ABL"]
CONFIGS  = ["WordInsert+Random","WordInsert+CBS","InsertSent+Random","InsertSent+CBS"]
COLORS   = [PALETTE["random"], PALETTE["cbs"], "#7fb8de", "#f09b9b"]

fig, ax = plt.subplots(figsize=(8.4, 3.6))
x = np.arange(len(DEFENSES)); w = 0.2
for i,(cfg,color) in enumerate(zip(CONFIGS, COLORS)):
    ax.bar(x + (i-1.5)*w, [E7[cfg][d] for d in DEFENSES], w, color=color, label=cfg)
ax.set_xticks(x); ax.set_xticklabels(["None","ONION","SpecSig","STRIP","ABL"])
ax.set_ylabel("ASR after defense (%)"); ax.set_ylim(0,105)
ax.set_title("SST-2: ASR remaining after each defense (flag -> remove -> retrain)")
ax.legend(fontsize=7.5, ncol=2)
plt.tight_layout(); plt.savefig("fig_sst2_defenses.png", dpi=200, bbox_inches="tight"); plt.show()

print("Percentage points of ASR removed, relative to each config's own baseline:")
for cfg in CONFIGS:
    base = E7[cfg]["No defense"]
    drops = "  ".join("{}:{:+6.1f}".format(d.split()[0][:7], base-E7[cfg][d]) for d in DEFENSES[1:])
    print("  {:20s} base={:5.1f}   {}".format(cfg, base, drops))

## Summary table for this dataset

In [ ]:
rows = [{"stage":"E1 clean baseline","trigger":"-","method":"-",
          "CACC":E1["CACC"],"ASR":None,"ASR_negctrl":None}]
for trig in ("word","sent"):
    rows.append(dict(stage="E2 teacher", trigger=trig, method="random",
                     **{k:E2[trig][k] for k in ("CACC","ASR","ASR_negctrl")}))
    rows.append(dict(stage="E3 teacher", trigger=trig, method="cbs",
                     **{k:E3[trig][k] for k in ("CACC","ASR","ASR_negctrl")}))
rows.append({"stage":"E4 clean student","trigger":"-","method":"-",
              "CACC":E4["CACC"],"ASR":E4["ASR"],"ASR_negctrl":None})
for trig in ("word","sent"):
    rows.append(dict(stage="E5 student", trigger=trig, method="random",
                     **{k:E5[trig][k] for k in ("CACC","ASR","ASR_negctrl")}))
    rows.append(dict(stage="E6 student", trigger=trig, method="cbs",
                     **{k:E6[trig][k] for k in ("CACC","ASR","ASR_negctrl")}))

summary = pd.DataFrame(rows)
for c in ("CACC","ASR","ASR_negctrl"):
    summary[c] = (summary[c].astype(float)*100).round(2)
summary.to_csv("summary_sst2.csv", index=False)
print("saved summary_sst2.csv")
summary